In [1]:
import os

In [2]:
%pwd

'/home/user/Documents/Text_Summarization/research'

In [3]:
os.chdir("..")

In [4]:
%pwd

'/home/user/Documents/Text_Summarization'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: float
    gradient_accumulation_steps: int

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
          root_dir=config.root_dir,
          data_path=config.data_path,
          model_ckpt=config.model_ckpt,
          num_train_epochs=params.num_train_epochs,
          warmup_steps=params.warmup_steps,
          per_device_train_batch_size=params.per_device_train_batch_size,
          weight_decay=params.weight_decay,
          logging_steps=params.logging_steps,
           evaluation_strategy=params.evaluation_strategy,
           eval_steps=params.eval_steps,  
           save_steps=params.save_steps,
           gradient_accumulation_steps=params.gradient_accumulation_steps
)

        return model_trainer_config

In [8]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

/home/user/Documents/Text_Summarization/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config


    
    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)
        
        dataset_samsum_pt = load_from_disk(self.config.data_path)

        print("Before reduction:", len(dataset_samsum_pt["train"]))

        dataset_samsum_pt["train"] = dataset_samsum_pt["train"].select(range(100))
        dataset_samsum_pt["validation"] = dataset_samsum_pt["validation"].select(range(20))

        print("After reduction:", len(dataset_samsum_pt["train"]))
        print("After reduction:", len(dataset_samsum_pt["validation"]))

        trainer_args = TrainingArguments(
          output_dir=str(self.config.root_dir),
          num_train_epochs=self.config.num_train_epochs,
          warmup_steps=self.config.warmup_steps,
          per_device_train_batch_size=self.config.per_device_train_batch_size,
          per_device_eval_batch_size=self.config.per_device_train_batch_size,
          weight_decay=self.config.weight_decay,
          logging_steps=self.config.logging_steps,
          evaluation_strategy=self.config.evaluation_strategy,
          eval_steps=self.config.eval_steps,
          save_steps=int(self.config.save_steps),
          gradient_accumulation_steps=self.config.gradient_accumulation_steps
         )
        trainer = Trainer(model=model_pegasus, args=trainer_args,
                  tokenizer=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_samsum_pt["train"], 
                eval_dataset=dataset_samsum_pt["validation"])
        
        trainer.train()

        model_pegasus.save_pretrained( os.path.join(self.config.root_dir, "flan-t5-small-model"))
        tokenizer.save_pretrained(os.path.join(self.config.root_dir,"tokenizer"))

        
        

In [10]:
try:
    config = ConfigurationManager()
    trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=trainer_config)
    model_trainer.train()
except Exception as e:
    print(e)

[2026-04-26 21:22:40,830: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-04-26 21:22:40,837: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-26 21:22:40,840: INFO: common: created directory at: artifacts]
[2026-04-26 21:22:40,843: INFO: common: created directory at: artifacts/model_trainer]
Before reduction: 100
After reduction: 100
After reduction: 20


/home/user/Documents/Text_Summarization/.venv/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
  0%|          | 0/50 [00:00<?, ?it/s]/home/user/Documents/Text_Summarization/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
 20%|██        | 10/50 [00:48<02:26,  3.67s/it]

{'loss': 1.8432, 'grad_norm': 8.838675498962402, 'learning_rate': 4e-05, 'epoch': 0.2}


 40%|████      | 20/50 [01:38<02:00,  4.01s/it]

{'loss': 1.8245, 'grad_norm': 9.416629791259766, 'learning_rate': 3e-05, 'epoch': 0.4}


...

{'loss': 1.9998, 'grad_norm': 7.027866840362549, 'learning_rate': 0.0, 'epoch': 1.0}


                                               
100%|██████████| 50/50 [03:50<00:00,  3.55s/it]

{'eval_loss': 1.6675268411636353, 'eval_runtime': 5.8125, 'eval_samples_per_second': 3.441, 'eval_steps_per_second': 3.441, 'epoch': 1.0}


100%|██████████| 50/50 [03:52<00:00,  4.65s/it]


{'train_runtime': 232.4235, 'train_samples_per_second': 0.43, 'train_steps_per_second': 0.215, 'train_loss': 1.8757218551635741, 'epoch': 1.0}
